In [26]:
print("Contenu du fichier output.log :")
with open('output.log', 'r') as f:
    print(f.read())


Contenu du fichier output.log :


2026-05-28 13:50:27.576 Port 8501 is not available
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [27]:
import pkg_resources

# Dépendances connues utilisées dans le script Streamlit
# Celles-ci devraient être dans votre requirements.txt
required_packages = [
    'streamlit',
    'plotly',
    'pandas'
]

# Créer le fichier requirements.txt
with open('requirements.txt', 'w') as f:
    for package in required_packages:
        try:
            # Obtenir la version exacte du package installé
            version = pkg_resources.get_distribution(package).version
            f.write(f"{package}=={version}\n")
        except pkg_resources.DistributionNotFound:
            print(f"Attention : Le package '{package}' n'est pas trouvé dans l'environnement actuel. Il sera ajouté sans version spécifique.")
            f.write(f"{package}\n")

print("✅ Le fichier `requirements.txt` a été créé avec succès !")

# Afficher le contenu pour vérification
with open('requirements.txt', 'r') as f:
    print("\nContenu de `requirements.txt` :")
    print(f.read())


✅ Le fichier `requirements.txt` a été créé avec succès !

Contenu de `requirements.txt` :
streamlit==1.57.0
plotly==5.24.1
pandas==2.2.2



/tmp/ipykernel_1873/4216945662.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [28]:
import subprocess
import time
import re

# ============================================================
# CELLULE 1 — Installation des dépendances (pour Colab)
# ============================================================
!pip install streamlit plotly pandas -q
!npm install -g localtunnel # Install localtunnel

print("✅ Dépendances installées et localtunnel prêt.")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦✅ Dépendances installées et localtunnel prêt.


In [29]:
# ============================================================
# CELLULE 2 — Lancement de l'app Streamlit via localtunnel
# ============================================================

# Contenu de l'application Streamlit (le code pur de l'app)
streamlit_app_code = """
import streamlit as st
import random
import pandas as pd
import plotly.express as px

# ── Configuration de la page ──────────────────────────────────
st.set_page_config(
    page_title="Rotation Culturale",
    page_icon="🌱",
    layout="wide"
)

# ── Custom CSS ────────────────────────────────────────────────
# You can customize the look of your app here.
# For example, let's change the background color of the main content area.
custom_css = '''
<style>
    /* Change the background color of the main content area */
    .stApp {
        background-color: #f0f2f6;
    }
    /* Example: Change header colors */
    h1, h2, h3, h4, h5, h6 {
        color: #1a5e1e;
    }
    /* Example: Change button style */
    .stButton>button {
        background-color: #4CAF50;
        color: white;
        border-radius: 5px;
        border: none;
        padding: 10px 20px;
        font-size: 16px;
    }
    .stButton>button:hover {
        background-color: #45a049;
    }
</style>
'''
st.markdown(custom_css, unsafe_allow_html=True)

cultures = ["Maïs", "Mil", "Arachide", "Haricot", "Sorgho"]
legumineuses = ["Arachide", "Haricot"]

# ── Classe environnement ──────────────────────────────────────
class EnvironnementAgricole:
    def __init__(self):
        self.sol = 100
        self.profit = 0
        self.annee = 1

    def appliquer_rotation(self, culture):
        rendement = random.randint(50, 120)

        if culture in legumineuses:
            self.sol = min(self.sol + 5, 150)   # les légumineuses régénèrent le sol
            bonus = 20
        else:
            self.sol -= 8                        # les céréales appauvrissent le sol
            bonus = 10

        climat = random.choice(["Bon", "Moyen", "Mauvais"])
        if climat == "Mauvais":
            rendement -= 20

        rendement = max(rendement, 0)            # rendement ne peut pas être négatif
        gain = rendement + bonus
        self.profit += gain

        # CORRECTION : incrémenter l'année AVANT de retourner le résultat
        annee_actuelle = self.annee
        self.annee += 1

        return {
            "annee": annee_actuelle,
            "culture": culture,
            "type": "Légumineuse" if culture in legumineuses else "Céréale",
            "rendement": rendement,
            "sol": self.sol,
            "climat": climat,
            "gain": gain,
            "profit_total": self.profit
        }

# ── Initialisation du state ───────────────────────────────────
if "env" not in st.session_state:
    st.session_state.env = EnvironnementAgricole()
if "historique" not in st.session_state:
    st.session_state.historique = []

env = st.session_state.env

# ── En-tête ───────────────────────────────────────────────────
st.title("🌱 Système de recommandation de rotation culturale durable")
st.caption("Simulation basée sur l'apprentissage par renforcement")

# ── Métriques en temps réel ───────────────────────────────────
col1, col2, col3 = st.columns(3)
col1.metric("📅 Année", env.annee)
col2.metric("🌍 Santé du sol", f"{env.sol}/100")
col3.metric("💰 Profit cumulé", env.profit)

# Barre de santé du sol
sol_pct = max(0, min(100, env.sol))
if sol_pct > 70:
    couleur = "normal"
elif sol_pct > 40:
    couleur = "off"
else:
    couleur = "inverse"
st.progress(sol_pct / 100, text=f"Santé du sol : {sol_pct}%")

st.divider()

# ── Panneau de simulation ─────────────────────────────────────
col_gauche, col_droite = st.columns([1, 2])

with col_gauche:
    st.subheader("🌾 Choix de la culture")

    culture_choisie = st.selectbox(
        "Sélectionner une culture :",
        cultures,
        help="Les légumineuses (Arachide, Haricot) régénèrent le sol."
    )

    # Conseil intelligent
    if culture_choisie in legumineuses:
        st.success("✅ Bon choix ! Cette légumineuse régénère le sol (+5).")
    else:
        if env.sol < 50:
            st.warning("⚠️ Sol appauvri. Une légumineuse serait préférable.")
        else:
            st.info("ℹ️ Culture céréalière. Impact sur le sol : -8.")

    lancer = st.button("▶️ Lancer la simulation", use_container_width=True, type="primary")

    if lancer:
        resultat = env.appliquer_rotation(culture_choisie)
        st.session_state.historique.append(resultat)
        st.rerun()

    # Bouton reset
    if st.button("🔄 Réinitialiser", use_container_width=True):
        st.session_state.env = EnvironnementAgricole()
        st.session_state.historique = []
        st.rerun()

with col_droite:
    st.subheader("📊 Résultats et graphiques")

    if st.session_state.historique:
        df = pd.DataFrame(st.session_state.historique)

        # Dernier résultat
        dernier = st.session_state.historique[-1]
        st.markdown("**Dernière simulation — Année {}**".format(dernier['annee']))
        r1, r2, r3 = st.columns(3)
        r1.metric("Culture", dernier["culture"])
        r2.metric("Climat", dernier["climat"])
        r3.metric("Rendement", dernier["rendement"])

        # Alerte sol critique
        if env.sol <= 40:
            st.error("🚨 Sol critique ! Utilisez une légumineuse immédiatement.")
        elif env.sol <= 60:
            st.warning("⚠️ Sol appauvri. Pensez à alterner avec Arachide ou Haricot.")

        # Graphique évolution du sol
        fig_sol = px.line(
            df, x="annee", y="sol",
            title="Évolution de la santé du sol",
            labels={"annee": "Année", "sol": "Indice sol"},
            color_discrete_sequence=["#3B6D11"],
            markers=True
        )
        fig_sol.add_hline(y=40, line_dash="dash", line_color="red", annotation_text="Seuil critique")
        fig_sol.update_layout(
            height=250,
            margin=dict(t=40, b=20),
            hovermode="x unified" # Améliore l'interaction au survol
        )
        fig_sol.update_traces(
            hovertemplate="<b>Année:</b> %{x}<br><b>Santé du sol:</b> %{y}<extra></extra>"
        )
        st.plotly_chart(fig_sol, use_container_width=True)

        # Graphique profit cumulé
        fig_profit = px.bar(
            df, x="annee", y="gain",
            title="Gain par année",
            labels={"annee": "Année", "gain": "Gain"},
            color="type",
            color_discrete_map={"Légumineuse": "#639922", "Céréale": "#185FA5"}
        )
        fig_profit.update_layout(
            height=250,
            margin=dict(t=40, b=20),
            hovermode="x unified" # Améliore l'interaction au survol
        )
        fig_profit.update_traces(
            hovertemplate="<b>Année:</b> %{x}<br><b>Type:</b> %{customdata[0]}<br><b>Gain:</b> %{y}<extra></extra>",
            customdata=df[['type']]
        )
        st.plotly_chart(fig_profit, use_container_width=True)

    else:
        st.info("Lance une simulation pour voir les résultats et les graphiques.")

# ── Prédictions futures ───────────────────────────────────────
if st.session_state.historique:
    st.divider()
    st.subheader("🔮 Prédictions des prochaines années")

    # Exemple simple de prédiction: projection linéaire basée sur la tendance des 3 dernières années
    # Pour une prédiction réelle, vous intégreriez un modèle ML/AI ici.
    historique_df = pd.DataFrame(st.session_state.historique)
    last_annee = historique_df['annee'].max()
    future_years = [last_annee + i for i in range(1, 6)] # Prédiction pour les 5 prochaines années

    # Calculer la tendance moyenne du sol et du profit sur les dernières années
    if len(historique_df) >= 3:
        recent_data = historique_df.tail(3)
        avg_sol_change = (recent_data['sol'].iloc[-1] - recent_data['sol'].iloc[0]) / (len(recent_data) - 1) if len(recent_data) > 1 else 0
        avg_profit_change = (recent_data['profit_total'].iloc[-1] - recent_data['profit_total'].iloc[0]) / (len(recent_data) - 1) if len(recent_data) > 1 else 0
    else:
        # If not enough historical data, use the last value as a base without change
        avg_sol_change = 0
        avg_profit_change = 0

    predicted_sol = []
    predicted_profit = []
    current_sol = historique_df['sol'].iloc[-1] if not historique_df.empty else 100
    current_profit = historique_df['profit_total'].iloc[-1] if not historique_df.empty else 0

    for year in future_years:
        current_sol += avg_sol_change # Appliquer la tendance
        current_profit += avg_profit_change # Appliquer la tendance
        predicted_sol.append(max(0, min(150, current_sol))) # Limiter la santé du sol
        predicted_profit.append(current_profit)

    # Créer un DataFrame pour les prédictions
    predictions_df = pd.DataFrame({
        'annee': future_years,
        'sol': predicted_sol,
        'profit_total': predicted_profit
    })

    # Combiner les données historiques et les prédictions pour le graphique
    combined_df = pd.concat([historique_df[['annee', 'sol', 'profit_total']], predictions_df], ignore_index=True)

    # Graphique de prédiction de la santé du sol
    fig_pred_sol = px.line(
        combined_df, x="annee", y="sol",
        title="Prédiction de l'Évolution de la Santé du Sol",
        labels={"annee": "Année", "sol": "Indice sol"},
        color_discrete_sequence=["#1A5E1E"],
        markers=True
    )
    fig_pred_sol.add_vline(x=last_annee + 0.5, line_dash="dash", line_color="grey", annotation_text="Début de la prédiction")
    fig_pred_sol.update_layout(
        height=300,
        margin=dict(t=40, b=20),
        hovermode="x unified"
    )
    fig_pred_sol.update_traces(
        hovertemplate="<b>Année:</b> %{x}<br><b>Santé du sol:</b> %{y}<extra></extra>"
    )
    st.plotly_chart(fig_pred_sol, use_container_width=True)

    # Graphique de prédiction du profit cumulé
    fig_pred_profit = px.line(
        combined_df, x="annee", y="profit_total",
        title="Prédiction du Profit Cumulé",
        labels={"annee": "Année", "profit_total": "Profit Cumulé"},
        color_discrete_sequence=["#4CAF50"],
        markers=True
    )
    fig_pred_profit.add_vline(x=last_annee + 0.5, line_dash="dash", line_color="grey", annotation_text="Début de la prédiction")
    fig_pred_profit.update_layout(
        height=300,
        margin=dict(t=40, b=20),
        hovermode="x unified"
    )
    fig_pred_profit.update_traces(
        hovertemplate="<b>Année:</b> %{x}<br><b>Profit Cumulé:</b> %{y}<extra></extra>"
    )
    st.plotly_chart(fig_pred_profit, use_container_width=True)


# ── Historique détaillé ───────────────────────────────────────
if st.session_state.historique:
    st.divider()
    st.subheader("📋 Historique complet")
    df_display = pd.DataFrame(st.session_state.historique)[[
        "annee", "culture", "type", "climat", "rendement", "sol", "gain", "profit_total"
    ]]
    df_display.columns = ["Année", "Culture", "Type", "Climat", "Rendement", "Sol", "Gain", "Profit cumulé"]
    st.dataframe(df_display, use_container_width=True, hide_index=True)

# ── À propos ──────────────────────────────────────────────────
with st.expander("ℹ️ À propos du projet"):
    st.markdown('''
**Objectif** : Recommander une rotation culturale durable en tenant compte de la santé
du sol, des conditions climatiques, du rendement agricole et du profit économique.

**Technologies** : Python · Streamlit · Plotly · Simulation stochastique

**Logique IA** :
- L'agriculteur est l'agent qui prend des décisions
- L'environnement répond avec un état (sol, climat, rendement)
- Les légumineuses régénèrent le sol → stratégie long terme
- Les céréales maximisent le gain court terme → compromis rendement/durabilité
    ''')
"""

# Écriture du fichier 'streamlit_app.py'
with open("streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_app_code)

print("✅ `streamlit_app.py` écrit avec succès.")

# Lancement de Streamlit en arrière-plan
subprocess.Popen(
    ["streamlit", "run", "streamlit_app.py", "--server.port", "8501",
     "--server.headless", "true"],
    stdout=open("output.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(4)  # Attendre que Streamlit démarre

print("\n🚀 Lancement de localtunnel pour exposer Streamlit sur le port 8501...")
print("\nClique sur le lien qui appara cit ci-dessous pour accéder à ton application Streamlit !")

# Création du tunnel localtunnel
get_ipython().system_raw('lt --port 8501 --subdomain streamapp > url.txt 2>&1 &')

# Attendre que localtunnel génère l'URL
time.sleep(5)  # Give localtunnel a bit more time to start up

with open('url.txt', 'r') as f:
    tunnel_url = f.read().strip()

# Check if the tunnel_url contains an actual URL (simple check for 'https')
if "https://" in tunnel_url:
    print(f"\n🌐 Lien de ton application Streamlit via localtunnel :\n👉 {tunnel_url}")
    print("(Ce lien est temporaire et valable tant que cette session Colab est active.)")
else:
    print("⚠️ Impossible d'obtenir l'URL de localtunnel. Vérifie la sortie pour les erreurs.")
    print("Contenu de url.txt :", tunnel_url)


✅ `streamlit_app.py` écrit avec succès.

🚀 Lancement de localtunnel pour exposer Streamlit sur le port 8501...

Clique sur le lien qui appara cit ci-dessous pour accéder à ton application Streamlit !

🌐 Lien de ton application Streamlit via localtunnel :
👉 your url is: https://perfect-pug-57.loca.lt
(Ce lien est temporaire et valable tant que cette session Colab est active.)


In [30]:
print(streamlit_app_code)


import streamlit as st
import random
import pandas as pd
import plotly.express as px

# ── Configuration de la page ──────────────────────────────────
st.set_page_config(
    page_title="Rotation Culturale",
    page_icon="🌱",
    layout="wide"
)

# ── Custom CSS ────────────────────────────────────────────────
# You can customize the look of your app here.
# For example, let's change the background color of the main content area.
custom_css = '''
<style>
    /* Change the background color of the main content area */
    .stApp {
        background-color: #f0f2f6;
    }
    /* Example: Change header colors */
    h1, h2, h3, h4, h5, h6 {
        color: #1a5e1e;
    }
    /* Example: Change button style */
    .stButton>button {
        background-color: #4CAF50;
        color: white;
        border-radius: 5px;
        border: none;
        padding: 10px 20px;
        font-size: 16px;
    }
    .stButton>button:hover {
        background-color: #45a049;
    }
</style>
'''
st.markdown(

In [31]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful